In [16]:

import tensorflow as tf

# FORCE EAGER EXECUTION (REQUIRED FOR QUANTUM SIMULATION)
tf.config.run_functions_eagerly(True)
tf.data.experimental.enable_debug_mode()

print("✅ TensorFlow eager execution forced ON")

✅ TensorFlow eager execution forced ON


In [ ]:
import os

# 🚨 CRITICAL FIX: Force TensorFlow to use Legacy Keras (Keras 2)
# This restores compatibility with PennyLane's KerasLayer
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
import pennylane as qml

# Check to ensure the fix worked
try:
    from pennylane.qnn import KerasLayer
    print("✅ PennyLane KerasLayer loaded successfully!")
except ImportError:
    print("❌ KerasLayer still missing. Please ensure you restarted the kernel.")

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pennylane as qml
import os
import time
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.saving import register_keras_serializable
from keras.optimizers import Optimizer
import pandas as pd
from datetime import datetime

In [20]:
@register_keras_serializable(package="Custom", name="AdamSGDHybrid")
class AdamSGDHybrid(keras.optimizers.Optimizer):

    def __init__(
        self,
        learning_rate=1e-4,          # 🔧 LOWER LR for quantum stability
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-7,
        momentum=0.6,
        mix_factor=0.5,              # 🔧 80% Adam, 20% SGD
        name="AdamSGDHybrid",
        **kwargs
    ):
        super().__init__(learning_rate=learning_rate, name=name, **kwargs)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.momentum = momentum
        self.mix_factor = mix_factor

    def build(self, var_list):
        self.m = [self.add_variable_from_reference(v, "m") for v in var_list]
        self.v = [self.add_variable_from_reference(v, "v") for v in var_list]
        self.sgd_mom = [self.add_variable_from_reference(v, "sgd_mom") for v in var_list]

    def update_step(self, grad, var, m, v, sgd_mom):
        lr = self.learning_rate

        m.assign(self.beta1 * m + (1.0 - self.beta1) * grad)
        v.assign(self.beta2 * v + (1.0 - self.beta2) * tf.square(grad))
        adam_update = m / (tf.sqrt(v) + self.epsilon)

        sgd_mom.assign(self.momentum * sgd_mom + grad)
        sgd_update = sgd_mom

        update = self.mix_factor * adam_update + (1.0 - self.mix_factor) * sgd_update
        var.assign_sub(lr * update)

    def get_config(self):
        config = super().get_config()
        config.update({
            "beta1": self.beta1,
            "beta2": self.beta2,
            "epsilon": self.epsilon,
            "momentum": self.momentum,
            "mix_factor": self.mix_factor,
        })
        return config


In [21]:
# # instantiate the hybrid optimizer for later use
# hybrid_optimizer = AdamSGDHybrid(
#     learning_rate=0.001,
#     beta1=0.9,
#     beta2=0.999,
#     momentum=0.9,
#     mix_factor=0.5   # 50% Adam + 50% SGD
# )

In [22]:
# reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_SIZE = 64
BATCH_SIZE = 16
N_QUBITS = 4
N_LAYERS = 2
EPOCHS = 30
K_FOLDS = 5
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

# print config (preserved)
print("="*70)
print("QUANTUM-INSPIRED CNN FOR BRAIN TUMOR CLASSIFICATION")
print("="*70)
print(f"\nConfiguration:")
print(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Quantum Qubits: {N_QUBITS}")
print(f"  Quantum Layers: {N_LAYERS}")
print(f"  K-Folds: {K_FOLDS}")
print(f"  Classes: {CLASS_NAMES}")
print("="*70)

QUANTUM-INSPIRED CNN FOR BRAIN TUMOR CLASSIFICATION

Configuration:
  Image Size: 64x64
  Quantum Qubits: 4
  Quantum Layers: 2
  K-Folds: 5
  Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [23]:
@register_keras_serializable(package="Custom", name="QuantumKerasLayer")
class QuantumKerasLayer(layers.Layer):

    def __init__(self, n_qubits, n_layers, **kwargs):
        super().__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        self.dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="numpy", diff_method="parameter-shift")
        def circuit(inputs, weights):
            qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
            qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qnode = circuit

    def build(self, input_shape):
        self.q_weights = self.add_weight(
            shape=(self.n_layers, self.n_qubits, 3),
            initializer=tf.random_normal_initializer(0.05),
            trainable=True,
            name="q_weights"
        )

    def call(self, inputs):

        def quantum_forward(x, w):
            out = self.qnode(x, w)
            return np.array(out, dtype=np.float32)

        outputs = tf.map_fn(
            lambda x: tf.py_function(
                func=quantum_forward,
                inp=[x, self.q_weights],
                Tout=tf.float32
            ),
            inputs,
            fn_output_signature=tf.TensorSpec(
                shape=(self.n_qubits,), dtype=tf.float32
            )
        )

        return outputs


    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_qubits)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "n_qubits": self.n_qubits,
            "n_layers": self.n_layers
        })
        return cfg


In [24]:
# Cell 46: Updated Quantum Circuit Definition

import pennylane as qml
import tensorflow as tf

# Define the device
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="tf", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Encode classical data (Angle Embedding is standard for this)
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    
    # Variational Quantum Layers
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    
    # Measure Z expectation on each qubit
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

# Define the weight shapes for the KerasLayer
# StronglyEntanglingLayers expects shape (n_layers, n_qubits, 3)
weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 3)}

print("✅ Quantum Circuit & QNode configured successfully.")

✅ Quantum Circuit & QNode configured successfully.


In [25]:
@register_keras_serializable(package="Custom", name="QuantumInspiredLayer")
class QuantumInspiredLayer(layers.Layer):
    """
    Quantum-inspired layer that emulates quantum behavior with classical operations.
    This is a classical fallback layer (NOT a quantum simulator),
    kept only for safety if PennyLane simulation fails.
    """
    def __init__(self, n_qubits, n_layers, **kwargs):
        super().__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_layers = n_layers

    def build(self, input_shape):
        self.rotation_weights = self.add_weight(
            name="rotation_weights",
            shape=(self.n_layers, self.n_qubits, 2),
            initializer=keras.initializers.RandomUniform(0, 2 * np.pi),
            trainable=True
        )

        self.interaction_weights = self.add_weight(
            name="interaction_weights",
            shape=(self.n_layers, self.n_qubits, self.n_qubits),
            initializer=keras.initializers.GlorotUniform(),
            trainable=True
        )

        super().build(input_shape)

    def call(self, inputs):
        x = inputs

        for layer in range(self.n_layers):
            # RY approx
            ry_transform = (
                tf.cos(self.rotation_weights[layer, :, 0]) * x +
                tf.sin(self.rotation_weights[layer, :, 0]) * tf.roll(x, shift=1, axis=-1)
            )

            # RZ approx
            rz_transform = (
                tf.cos(self.rotation_weights[layer, :, 1]) * ry_transform -
                tf.sin(self.rotation_weights[layer, :, 1]) * tf.roll(ry_transform, shift=-1, axis=-1)
            )

            # Interaction matrix
            interaction = tf.matmul(
                tf.expand_dims(rz_transform, axis=-1),
                tf.expand_dims(rz_transform, axis=-2)
            )

            interaction_effect = tf.reduce_sum(
                interaction * self.interaction_weights[layer],
                axis=-1
            )

            x = tf.tanh(rz_transform + 0.1 * interaction_effect)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "n_qubits": self.n_qubits,
            "n_layers": self.n_layers,
        })
        return config

In [ ]:
# Cell 48: Updated Model Creation

def create_quantum_cnn_model(use_real_quantum=True):
    """
    Hybrid Classical–Quantum CNN Model
    """
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # --- Classical Feature Extraction ---
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # --- Quantum Interface ---
    # Downscale to N_QUBITS to match quantum width
    x = layers.Dense(N_QUBITS, activation='tanh')(x)
    
    if use_real_quantum:
        # Use the OFFICIAL PennyLane KerasLayer
        # output_dim must match the number of measurements returned by the circuit
        x = qml.qnn.KerasLayer(quantum_circuit, weight_shapes, output_dim=N_QUBITS)(x)
        
    else:
        # Fallback to classical approximation
        x = QuantumInspiredLayer(N_QUBITS, N_LAYERS)(x)
        
    # --- Post-Quantum Classification ---
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
    
    return keras.Model(inputs=inputs, outputs=outputs)

print("✅ Model creator updated to use qml.qnn.KerasLayer.")

✅ Model creator updated to use qml.qnn.KerasLayer.


In [27]:
# Metrics logging callback that saves per-epoch metrics (and time) to CSV
@register_keras_serializable(package="Custom", name="MetricsLogger")
class MetricsLogger(keras.callbacks.Callback):
    def __init__(self, fold_index, csv_path="training_metrics.csv"):
        super().__init__()
        self.fold_index = fold_index
        self.csv_path = csv_path
        self.epoch_start_time = None
        self.write_header = not os.path.exists(csv_path)

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.time() - self.epoch_start_time

        row = {
            "Optimizer": "HybridAdamSDG",
            "ModelType": "FEDERATED CNN",
            "Round_Fold": self.fold_index,
            "Client": 1,
            "Epoch": epoch + 1,
            "TimeSec": epoch_time,
            "TrainAcc": logs.get("accuracy"),
            "ValAcc": logs.get("val_accuracy"),
            "TrainLoss": logs.get("loss"),
            "ValLoss": logs.get("val_loss"),
            "LR": float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))

        }

        df = pd.DataFrame([row])
        df.to_csv(
            self.csv_path,
            mode="w" if self.write_header else "a",
            header=self.write_header,
            index=False
        )
        self.write_header = False

    # ✅ REQUIRED FOR register_keras_serializable
    def get_config(self):
        config = super().get_config()
        config.update({
            "fold_index": self.fold_index,
            "csv_path": self.csv_path
        })
        return config

In [28]:
def load_data(data_dir, img_size=IMG_SIZE):
    """Load and preprocess images"""
    images = []
    labels = []
    
    for idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Warning: {class_dir} not found, skipping...")
            continue
            
        print(f"Loading {class_name}...", end=" ")
        count = 0
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            try:
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    img = img / 255.0  # Normalize
                    images.append(img)
                    labels.append(idx)
                    count += 1
            except Exception as e:
                continue
        print(f"{count} images loaded")
    
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)


def visualize_quantum_layer(model, test_images, save_path='quantum_layer_viz.png'):
    """Visualize quantum layer activations"""
    quantum_model = keras.Model(
        inputs=model.input,
        outputs=model.get_layer('quantum_layer').output
    )
    
    n_samples = min(4, len(test_images))
    indices = np.random.choice(len(test_images), n_samples, replace=False)
    
    fig, axes = plt.subplots(n_samples, 2, figsize=(12, 3*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, 2)
    
    for idx, img_idx in enumerate(indices):
        img = test_images[img_idx:img_idx+1]
        
        # Get quantum layer activation
        quantum_output = quantum_model.predict(img, verbose=0)[0]
        
        # Original image
        axes[idx, 0].imshow(img[0])
        axes[idx, 0].set_title('Input Image', fontsize=10)
        axes[idx, 0].axis('off')
        
        # Quantum activation
        axes[idx, 1].bar(range(N_QUBITS), quantum_output)
        axes[idx, 1].set_title('Quantum Layer Activations', fontsize=10)
        axes[idx, 1].set_xlabel('Qubit')
        axes[idx, 1].set_ylabel('Activation')
        axes[idx, 1].set_ylim([-1, 1])
        axes[idx, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Quantum layer visualization saved to {save_path}")
    plt.close()

def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix - Test Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Confusion matrix saved to {save_path}")
    plt.close()

def plot_training_history(histories, save_path='training_history.png'):
    """Plot training history across folds"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    for fold_idx, history in enumerate(histories):
        axes[0].plot(history['accuracy'], label=f'Fold {fold_idx+1} Train', alpha=0.6)
        axes[0].plot(history['val_accuracy'], label=f'Fold {fold_idx+1} Val', alpha=0.6, linestyle='--')
    
    axes[0].set_title('Model Accuracy Across Folds')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    for fold_idx, history in enumerate(histories):
        axes[1].plot(history['loss'], label=f'Fold {fold_idx+1} Train', alpha=0.6)
        axes[1].plot(history['val_loss'], label=f'Fold {fold_idx+1} Val', alpha=0.6, linestyle='--')
    
    axes[1].set_title('Model Loss Across Folds')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Training history saved to {save_path}")
    plt.close()

In [29]:
def main():
    train_dir = '/home/hassan/QML/Brain_Tumor_data/Training'
    test_dir = '/home/hassan/QML/Brain_Tumor_data/Testing'
    metrics_csv = "training_metrics.csv"
    
    if not os.path.exists(train_dir):
        print(f"\nError: Training directory '{train_dir}' not found!")
        print("Please ensure the dataset is in the correct structure.")
        return
    
    # Load data
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)
    X_train, y_train = load_data(train_dir)
    print(f"\nTraining data: {X_train.shape}, Labels: {y_train.shape}")
    
    if os.path.exists(test_dir):
        X_test, y_test = load_data(test_dir)
        print(f"Testing data: {X_test.shape}, Labels: {y_test.shape}")
    else:
        print(f"\nWarning: Test directory '{test_dir}' not found.")
        X_test, y_test = None, None
    
    # K-Fold Cross Validation
    print("\n" + "="*70)
    print("K-FOLD CROSS VALIDATION TRAINING")
    print("="*70)
    
    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    histories = []
    best_model = None
    best_val_acc = 0
    
    # remove old metrics file if exists (optional — comment out if you prefer to append across runs)
    if os.path.exists(metrics_csv):
        os.remove(metrics_csv)
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
        print(f"\n{'='*70}")
        print(f"FOLD {fold + 1}/{K_FOLDS}")
        print('='*70)
        
        X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
        
        print(f"Train samples: {len(X_fold_train)}, Val samples: {len(X_fold_val)}")
        
        # Create and compile model
        # IMPORTANT: use the hybrid_optimizer defined earlier to actually use your custom optimizer
        tf.keras.backend.clear_session()

        optimizer = AdamSGDHybrid(
            learning_rate=0.001,
            beta1=0.9,
            beta2=0.999,
            momentum=0.9,
            mix_factor=0.5
        )

        model = create_quantum_cnn_model(use_real_quantum=True)

        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
            run_eagerly=True
        )

        
        # Callbacks include the new MetricsLogger which writes per-epoch rows to CSV
        metrics_logger = MetricsLogger(fold_index=fold+1, csv_path=metrics_csv)
        callbacks = [
            keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
            metrics_logger
        ]
        
        # Train
        start_time = time.time()
        try:
            history = model.fit(
                X_fold_train, y_fold_train,
                validation_data=(X_fold_val, y_fold_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=callbacks,
                verbose=1
            )
            training_time = time.time() - start_time
            
            val_loss, val_acc = model.evaluate(X_fold_val, y_fold_val, verbose=0)
            fold_scores.append(val_acc)
            histories.append(history.history)
            
            print(f"\nFold {fold+1} Results:")
            print(f"  Validation Accuracy: {val_acc:.4f}")
            print(f"  Validation Loss: {val_loss:.4f}")
            print(f"  Training Time: {training_time:.2f} seconds")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                print(f"  *** New best model! ***")
                best_model.save(f"quantum_cnn_best_model_fold_{fold+1}.keras")
        
        except Exception as e:
            print(f"\nError in fold {fold+1}: {str(e)}")
            continue
    
    if len(fold_scores) == 0:
        print("\nNo folds completed successfully.")
        return
    
    # Cross-validation results
    print("\n" + "="*70)
    print("CROSS-VALIDATION RESULTS")
    print("="*70)
    print(f"Fold Accuracies: {[f'{acc:.4f}' for acc in fold_scores]}")
    print(f"Mean CV Accuracy: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
    print(f"Best Fold Accuracy: {max(fold_scores):.4f}")
    
    # Plot training history
    if len(histories) > 0:
        plot_training_history(histories)
    
    # Test on test set
    if X_test is not None and y_test is not None and best_model is not None:
        print("\n" + "="*70)
        print("TEST SET EVALUATION")
        print("="*70)
        
        test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        
        # Predictions
        y_pred = best_model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        # Classification report
        print("\n" + "-"*70)
        print("CLASSIFICATION REPORT")
        print("-"*70)
        print(classification_report(y_test, y_pred_classes, target_names=CLASS_NAMES))
        
        # Detailed metrics
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred_classes, average=None, labels=range(len(CLASS_NAMES))
        )
        
        print("\n" + "-"*70)
        print("PER-CLASS PERFORMANCE ANALYSIS")
        print("-"*70)
        for i, class_name in enumerate(CLASS_NAMES):
            print(f"\n{class_name.upper()}:")
            print(f"  Precision: {precision[i]:.4f}")
            print(f"  Recall: {recall[i]:.4f}")
            print(f"  F1-Score: {f1[i]:.4f}")
            print(f"  Support: {support[i]}")
        
        # Visualizations
        plot_confusion_matrix(y_test, y_pred_classes)
        
        print("\n" + "-"*70)
        print("GENERATING VISUALIZATIONS")
        print("-"*70)
        visualize_quantum_layer(best_model, X_test)
    
    # Save model
    if best_model is not None:
        best_model.save('quantum_cnn_final_model.keras')
        print("\n" + "="*70)
        print("Model saved as 'quantum_cnn_final_model.keras'")
        print("="*70)
    
    print("\n✓ Training and evaluation complete!")
    print("\nGenerated files (examples):")
    print("  1. quantum_cnn_final_model.keras - Trained model")
    print("  2. training_history.png - Training curves")
    print("  3. confusion_matrix.png - Confusion matrix")
    print("  4. quantum_layer_viz.png - Quantum layer activations")
    print(f"  5. {metrics_csv} - Per-epoch metrics CSV (fold, epoch, times, loss, accuracy, val_loss, val_accuracy)")    


In [30]:
if __name__ == "__main__":
    main()


LOADING DATA
Loading glioma... 1321 images loaded
Loading meningioma... 1339 images loaded
Loading notumor... 1595 images loaded
Loading pituitary... 1457 images loaded

Training data: (5712, 64, 64, 3), Labels: (5712,)
Loading glioma... 300 images loaded
Loading meningioma... 306 images loaded
Loading notumor... 405 images loaded
Loading pituitary... 300 images loaded
Testing data: (1311, 64, 64, 3), Labels: (1311,)

K-FOLD CROSS VALIDATION TRAINING

FOLD 1/5
Train samples: 4569, Val samples: 1143


AttributeError: module 'pennylane.qnn' has no attribute 'KerasLayer'